# 03. Угрозы №10 и №16: лазерное воздействие / прожекторы и световое загрязнение

| № | Угроза | Эффект | Последствия для миссии |
|---|---|---|---|
| 10 | Лазерное шоу / прожекторы (световое воздействие) | Выгорание пикселей матрицы, ослепление, схлопывание автоэкспозиции | Потеря визуального канала, нет данных |
| 16 | Световое загрязнение (городская ночная засветка) | Оптический шум, блики, снижение контраста | Ложные срабатывания детекторов, потеря ориентации |

## Связь с ТЗ
| Сигнал | Источник | Как используем |
|---|---|---|
| видеопоток (RGB) | вне потоков ТЗ (подтверждён заказчиком) | доля насыщенных пикселей, блобы, ореол, шум |
| тепловизионный канал | то же | подтверждение «настоящего» очага (лазер и городские огни тепловых аномалий не дают) |
| усиление/экспозиция ISP | **допущение**: метаданные камеры доступны | признак схлопывания автоэкспозиции |
| `timestamp`, `lat`, `lon` | БВС-1 | высота Солнца → ночь/день (контекстный приор) |
| выход | БВС-3 (алерт), НСУ-2 (зона), НСУ-4/5 | пометка кадров, зона «оптической опасности», маршрут |

## Идея
1. **Угроза 10.** Лазер даёт *резкий* прирост насыщенных пикселей, мерцает (импульсный режим) и не имеет теплового следа. Это отличает его от очага огня (есть тепловая аномалия), от городских огней (не меняются скачком) и от солнечного блика (не мерцает, днём). Строим zero-shot правила с последовательным добавлением этих признаков и сравниваем с одноклассовыми моделями (Isolation Forest, PCA-автоэнкодер).
2. **Угроза 16.** Оцениваем уровень засветки (индекс LPI) по признакам кадра и экспозиции и проверяем, как засветка порождает **ложные «очаги» в RGB**; сравниваем меры: тепловое подтверждение, априорная карта известных огней.

> **Допущения:** синтетический одноканальный кадр (цвет не моделируется), идеальная регистрация RGB/тепловизор, модель автоэкспозиции и ореола заданы нами. Реальных данных по лазерам и бликам нет: результат показывает работоспособность метода и компромиссы, а не точность на борту.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.ndimage import gaussian_filter, binary_dilation
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from skimage import exposure

from src.synth_light import SeqConfig, make_sequence, FPS, WARMUP, G_MIN
from src.light_features import seq_features, frame_features, noise_sigma, solar_elevation
from src.light_detectors import (rule_laser, AnomalyModels, rgb_fire_candidates, persistent, seq_event_outcome)
from src.light_eval import (benign_cfg, laser_cfg, build, evaluate_detectors, frame_auroc, run_detectors, DETECTORS, WARM, BENIGN_TYPES)
plt.rcParams["figure.dpi"] = 110

## 1. Симулятор камеры и сцен
Мир неподвижен, БВС смещается на 0,5 px/кадр (частота 5 Гц, кадр 96×96). Яркость сцены: фон × текстура + городские огни (ядро + степенной ореол) + очаг огня + солнечный блик + лазер. Камера: `x = g·L`, дробовой шум (Пуассон) + шум считывания, обрезка, гамма 1/2.2; **автоэкспозиция** подстраивает `g` по средней яркости кадра. Тепловизор радиометрический (К), зависит только от фона и очагов.

In [ ]:
cases = [("ночь, чисто", SeqConfig(seed=11)), ("ночь, город u=0.7", SeqConfig(urban=.7, seed=12)),
         ("ночь, лазер (2 пятна)", SeqConfig(laser=True, laser_I=1000, laser_spots=2, laser_duty=1.0, seed=13)),
         ("ночь, очаг огня", SeqConfig(fire=True, fire_I=60, seed=14)),
         ("день, блик на воде", SeqConfig(night=False, glint=True, seed=15)), ("день, чисто", SeqConfig(night=False, seed=16))]
fig, axs = plt.subplots(2, 6, figsize=(13, 4.6))
for j, (name, cfg) in enumerate(cases):
    s = make_sequence(cfg); t = 50
    axs[0, j].imshow(s["y"][t], cmap="gray", vmin=0, vmax=1); axs[0, j].set_title(f"{name}\ng={s['gain'][t]:.1f}", fontsize=8)
    axs[1, j].imshow(s["T"][t], cmap="inferno"); axs[0, j].axis("off"); axs[1, j].axis("off")
axs[0, 0].text(-0.05, 0.5, "RGB", transform=axs[0, 0].transAxes, rotation=90, va="center", ha="right")
axs[1, 0].text(-0.05, 0.5, "тепловизор", transform=axs[1, 0].transAxes, rotation=90, va="center", ha="right")
plt.tight_layout(); plt.show()

## 2. Признаки кадра
`sat_frac` (доля насыщенных пикселей), `amax`/`nbl` (площадь крупнейшего блоба, число блобов), `mean/p05/p99`, `log_gain` (усиление), `sigma` (шум по Иммеркеру [Immerkær, 1996], вне засветки), `halo` (площадь почти-насыщения относительно насыщения), `Tex_blob` (тепловой избыток в зоне засветки), а также временные: приращения относительно медианы прошлого (окно 4..25 кадров назад) и `n_down12` - число заметных *спадов* насыщения за 12 кадров (признак мерцания).

In [ ]:
ex = {"лазер (импульсный)": SeqConfig(laser=True, laser_I=300, laser_duty=0.6, seed=21, urban=0.3),
      "город u=0.7": SeqConfig(urban=.7, seed=22), "очаг огня": SeqConfig(fire=True, fire_I=60, seed=23),
      "блик (день)": SeqConfig(night=False, glint=True, seed=24)}
fig, axs = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
for name, cfg in ex.items():
    F = seq_features(make_sequence(cfg))
    axs[0].plot(F.sat_frac, label=name); axs[1].plot(F.log_gain); axs[2].plot(F.n_down12); axs[3].plot(F.Tex_blob)
axs[0].set_ylabel("sat_frac"); axs[0].set_yscale("symlog", linthresh=1e-3); axs[0].legend(fontsize=8, ncol=2)
axs[1].set_ylabel("ln(усиление)"); axs[2].set_ylabel("n_down12"); axs[3].set_ylabel("Tex_blob, К"); axs[3].set_xlabel("кадр")
for a in axs: a.grid(alpha=.3); a.axvspan(35, 65, color="gray", alpha=.08)
plt.tight_layout(); plt.show()

## 3. Угроза №10: обнаружение лазерного воздействия

### Детекторы (от простого к сложному)
| Имя | Правило / модель |
|---|---|
| **R0** `sat_jump` | скачок `sat_frac` над недавней базой: `max(sat[t-2..t]) − median(база) > thr + k·σ_база` (адаптивный порог; база замораживается при тревоге), удержание 6 кадров |
| **R1** `+thermal` | R0, но исключить зоны с тепловой аномалией ≥ 8 К (очаги огня) |
| **R2** `+pulse` | R1 + требование мерцания (≥ 1 спад насыщения) *или* очень большой засветки |
| **R3** `+day` | R2 + контекстный приор: днём порог выше (по высоте Солнца из `timestamp, lat, lon`) |
| **IForest** | Isolation Forest, обученный только на «безобидных» последовательностях |
| **PCA-AE** | ошибка восстановления PCA (линейный автоэнкодер) на тех же данных |

«Безобидные» последовательности: ночь чисто, ночной город, день чисто, блик, очаг огня ночью и днём. Лазерные: пятна 1-3, мерцание/непрерывные (20 %), яркость 10-3000 (отн.), ядро 0,8-3 px, ночь/день. **Три независимых набора зёрен:** обучение (1000+), подбор порога (4000+), финальная оценка (2000+/3000+).

In [ ]:
rng = np.random.default_rng(1000)
train = build([benign_cfg(rng, 1000 + i) for i in range(200)])
am = AnomalyModels(seed=0).fit(pd.concat([x[2] for x in train]))
rng = np.random.default_rng(4000)
tune_b = build([benign_cfg(rng, 4000 + i) for i in range(100)])
tune_l = build([("laser", laser_cfg(rng, 5000 + i)) for i in range(80)])

rows = []
for k in (2, 3, 5, 8, 12):
    fa, rec = {}, []
    for kind, cfg, F, lab in tune_b:
        fa.setdefault(kind, []).append(bool(rule_laser(F, k_sd=k)[0][WARM:].any()))
    for kind, cfg, F, lab in tune_l:
        rec.append(seq_event_outcome(rule_laser(F, k_sd=k)[0], lab["laser"], cfg.laser_onset)[0])
    nonglint = np.mean([v for kk, vv in fa.items() if kk != "day_glint" for v in vv])
    rows.append(dict(k_sd=k, recall=np.mean(rec), FA_without_glint=nonglint, FA_glint=np.mean(fa["day_glint"])))
tune_tab = pd.DataFrame(rows).round(3); display(tune_tab)
K_SD = float(tune_tab[tune_tab.FA_without_glint <= 0.01].k_sd.min()); print("выбрано k_sd =", K_SD)

In [ ]:
rng = np.random.default_rng(2000)
test_b = build([benign_cfg(rng, 2000 + i) for i in range(150)])
test_l = build([("laser", laser_cfg(rng, 3000 + i)) for i in range(150)])
res = evaluate_detectors(test_b + test_l, am, k_sd=K_SD)

L = res[res.laser]
tab = pd.DataFrame({
    "recall": L.groupby("det").flag.mean(),
    "recall импульс.": L[~L.cw].groupby("det").flag.mean(),
    "recall непрер.": L[L.cw].groupby("det").flag.mean(),
    "recall ночь": L[L.night].groupby("det").flag.mean(),
    "recall день": L[~L.night].groupby("det").flag.mean(),
    "медиана задержки, кадров": L[L.flag].groupby("det").delay.median()}).reindex(DETECTORS).round(2)
tab["медиана задержки, мс"] = (tab["медиана задержки, кадров"] * 1000 / FPS).round(0)
display(tab)
B = res[~res.laser]
fa_tab = B.groupby(["det", "kind"]).flag.mean().unstack().reindex(DETECTORS)[BENIGN_TYPES].round(2)
display(fa_tab)
au = pd.Series(frame_auroc(test_b + test_l, am, k_sd=K_SD)).round(3); display(au.to_frame("AUROC (кадры)").T)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
tab[["recall импульс.", "recall непрер."]].plot.bar(ax=axs[0], ylim=(0, 1.05)); axs[0].set_title("Полнота по типу лазера"); axs[0].tick_params(axis="x", rotation=30)
im = axs[1].imshow(fa_tab.values, vmin=0, vmax=1, cmap="Reds"); axs[1].set_title("Доля ложных тревог по типам безобидных сцен")
axs[1].set_xticks(range(6), BENIGN_TYPES, rotation=30, ha="right", fontsize=8); axs[1].set_yticks(range(6), DETECTORS, fontsize=8)
for (i, j), v in np.ndenumerate(fa_tab.values): axs[1].text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="k" if v < .6 else "w")
plt.tight_layout(); plt.show()

### Кривые по тяжести (яркость лазера) и ночь над городом
Тяжесть задаётся пиковой яркостью пятна `I` (ядро 1,5 px, 1 пятно, ночь, засветка города u ∈ [0; 0,6]).

In [ ]:
levels = [10, 30, 100, 300, 1000, 3000]
rng = np.random.default_rng(6000); sev_rows = []
for I in levels:
    items = build([("laser", laser_cfg(rng, 6000 + len(sev_rows) * 40 + i, I=I, night=True, core=1.5)) for i in range(30)])
    r = evaluate_detectors(items, am, k_sd=K_SD); r["I"] = I; sev_rows.append(r)
sev = pd.concat(sev_rows)
curve = sev.groupby(["I", "det"]).flag.mean().unstack()[DETECTORS]
display(curve.round(2))
ax = curve.plot(marker="o", figsize=(7, 4), logx=True); ax.set_xlabel("пиковая яркость пятна I (отн.)"); ax.set_ylabel("полнота"); ax.grid(alpha=.3)
ax.set_title("Полнота в зависимости от мощности лазера (ночь)"); plt.show()

### Смягчение: схлопывание автоэкспозиции
Яркое пятно поднимает среднюю яркость кадра, камера убирает усиление, и вся остальная сцена темнеет (потеря визуального канала без физического повреждения). Сравниваем меры:
- штатная автоэкспозиция (AE);
- AE по пикселям вне насыщения (маска);
- та же маска, расширенная на 6 и 12 px (учёт светлого ореола вокруг ядра);
- **блокировка усиления** через 2 кадра после начала события (имитация реакции на тревогу детектора).

Метрика: медианное усиление во время лазера / усиление до события (1 = экспозиция сохранена; <1 = фон «сжат»).

In [ ]:
VARS = {"штатная AE": {}, "маска насыщ.": dict(ae_mask_dilate=0), "маска +6 px": dict(ae_mask_dilate=6),
        "маска +12 px": dict(ae_mask_dilate=12), "блокировка (onset+2)": "freeze"}
ratios = {k: [] for k in VARS}; rng = np.random.default_rng(7000)
for i in range(40):
    cfg0 = laser_cfg(rng, 7000 + i, I=10 ** rng.uniform(2, 3.5), night=True, core=1.5)
    for k, v in VARS.items():
        cfg = SeqConfig(**{**cfg0.__dict__})
        if v == "freeze": cfg.ae_freeze_from = cfg.laser_onset + 2
        else:
            for a, b in v.items(): setattr(cfg, a, b)
        sq = make_sequence(cfg); g = sq["gain"]; ok = sq["labels"]["laser_pulse"]
        ratios[k].append(np.median(g[ok]) / np.median(g[15:cfg.laser_onset]))
mit = pd.DataFrame({k: [np.median(v), np.percentile(v, 10), np.percentile(v, 90)] for k, v in ratios.items()},
                   index=["медиана", "p10", "p90"]).T.round(2)
display(mit)
fig, ax = plt.subplots(figsize=(6.2, 3.4)); ax.boxplot([ratios[k] for k in VARS]); ax.set_xticks(range(1, 6), list(VARS), rotation=20, fontsize=8)
ax.axhline(1, c="k", ls=":"); ax.set_ylabel("усиление во время лазера / до"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

### Выход модуля (формат БВС-3 / НСУ-2)
Для лазерной тревоги дополнительно формируется запрос зоны «оптической опасности»: в ТЗ (НСУ-2) допустимы типы `no_fly_zone`, `obstacle_3d`, `terrain_patch`, поэтому используется `no_fly_zone` с ограниченным сроком действия (согласовать с заказчиком).

In [ ]:
def laser_alert(t, p, lat, lon, alt, uav_id="UAV-001"):
    return {"alert_id": f"opt-{int(t * 1000)}", "uav_id": uav_id,
            "alert_type": None,                     # TODO: в ТЗ нет типа для оптических угроз
            "severity": 4, "description": "OPTICAL_INTERFERENCE_LASER_SUSPECTED",
            "sensor_source": "rgb+thermal", "confidence": round(float(p), 2),
            "location": {"lat": lat, "lon": lon, "alt": alt}, "timestamp": int(t * 1000),
            "recommended_action": "lock_exposure_mask_saturation; switch_to_thermal; request_zone(no_fly_zone, ttl=900s)"}
laser_alert(12.4, 0.9, 50.11, 8.68, 120.0)

## 4. Угроза №16: световое загрязнение

### 4.1. Контекст «ночь / день» по телеметрии
Высота Солнца считается по `timestamp`, `lat`, `lon` (упрощённый алгоритм NOAA/Spencer). Порог ночи: Солнце ниже −6° (гражданские сумерки).

In [ ]:
print("контрольная точка: полдень по солнцу, Франкфурт, 21.06.2026 11:25 UTC ->", round(solar_elevation("2026-06-21 11:25", 50.11, 8.68), 1), "° (ожидаем ~63°)")
rows = []
for h in (0, 4, 7, 10, 13, 16, 17, 20):
    rows.append(dict(UTC=f"{h:02d}:00", июнь=round(solar_elevation(f"2026-06-21 {h:02d}:00", 50.11, 8.68), 1), ноябрь=round(solar_elevation(f"2026-11-06 {h:02d}:00", 50.11, 8.68), 1)))
sol = pd.DataFrame(rows).set_index("UTC"); sol["ночь (июнь)"] = sol["июнь"] < -6; sol["ночь (ноябрь)"] = sol["ноябрь"] < -6; display(sol)

### 4.2. Индекс засветки (LPI)
Истинный уровень засветки `u ∈ [0,1]` управляет числом и яркостью огней. Сравниваем:
- **LPI_g (zero-shot)** — по экспозиции: `LPI = (ln g_dark − ln g)/(ln g_dark − ln g_min)`, где `g_dark` — усиление камеры над тёмной местностью (калибровка на пролёте без огней), `g_min` — паспортный минимум усиления;
- **GBM-регрессия** на признаках кадра (`sat_frac, nbl, p05, sigma, mean, log_gain, halo`).

Проверка **смены домена**: обучение на ореоле `halo=0.01, r0=3`, тест на более сильном ореоле `halo=0.03, r0=5` (другая оптика/загрязнённый объектив).

In [ ]:
FE = ["log_gain", "sat_frac", "nbl", "p05", "sigma", "mean", "halo"]
def lpi_data(halo, r0, seed0, per=15):
    rows = []
    for u in np.linspace(0, 1, 6):
        for i in range(per):
            cfg = SeqConfig(urban=u, halo=halo, halo_r0=r0, seed=seed0 + int(u * 100) * 50 + i)
            F = seq_features(make_sequence(cfg), temporal=False).iloc[WARM:][FE].mean()
            rows.append(dict(u=u, **F))
    return pd.DataFrame(rows)
src_d = lpi_data(0.01, 3.0, 8000); tgt_d = lpi_data(0.03, 5.0, 9000)
lg_dark = src_d[src_d.u == 0].log_gain.mean()
lpi = lambda d: np.clip((lg_dark - d.log_gain) / (lg_dark - np.log(G_MIN)), 0, 1)
gbm = HistGradientBoostingRegressor(max_iter=200, random_state=0).fit(src_d[FE], src_d.u)
rows = []
for name, d in (("исходный домен (тест на новых сценах)", lpi_data(0.01, 3.0, 8500)), ("смена домена (сильный ореол)", tgt_d)):
    p = gbm.predict(d[FE])
    rows.append(dict(домен=name, Spearman_LPI=spearmanr(lpi(d), d.u)[0], Spearman_GBM=spearmanr(p, d.u)[0],
                     R2_GBM=r2_score(d.u, p), MAE_GBM=mean_absolute_error(d.u, p), средний_LPI_при_u0=lpi(d[d.u == 0]).mean(),
                     средний_LPI_при_u1=lpi(d[d.u == 1]).mean()))
lpi_tab = pd.DataFrame(rows).set_index("домен").round(3); display(lpi_tab)
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
for a, (name, d) in zip(axs, (("исходный", src_d), ("смена домена", tgt_d))):
    a.scatter(d.u, lpi(d), s=12, label="LPI_g (zero-shot)", alpha=.6); a.scatter(d.u, gbm.predict(d[FE]), s=12, label="GBM", alpha=.6)
    a.plot([0, 1], [0, 1], "k:"); a.set_xlabel("истинный уровень засветки u"); a.set_title(name); a.legend(fontsize=8); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### 4.3. Улучшение ночного кадра: что даёт и чем платит
Сравниваются гамма-подъём, многомасштабный Retinex (одномасштабный вариант, σ=15 px) и CLAHE. Метрики: σ шума (Иммеркер, вне засветки) и контраст (std яркости вне засветки). Ожидание: любое усиление повышает и контраст, и шум.

In [ ]:
s = make_sequence(SeqConfig(urban=0.5, seed=31)); y = s["y"][40].astype(float)
def msr(y, sig=15):
    r = np.log(y + 1e-3) - np.log(gaussian_filter(y, sig) + 1e-3); lo, hi = np.percentile(r, (1, 99)); return np.clip((r - lo) / (hi - lo), 0, 1)
variants = {"исходный": y, "гамма 0.6": y ** 0.6, "Retinex (SSR)": msr(y), "CLAHE": exposure.equalize_adapthist(y, clip_limit=0.02)}
valid = ~binary_dilation(y >= 0.98, iterations=2)
rows = [dict(вариант=k, шум_sigma=noise_sigma(v, valid), контраст_std=float(v[valid].std()), доля_насыщ=float((v >= .98).mean()))
        for k, v in variants.items()]
enh = pd.DataFrame(rows).set_index("вариант"); enh["контраст/шум"] = enh.контраст_std / enh.шум_sigma; display(enh.round(3))
fig, axs = plt.subplots(1, 4, figsize=(11, 3))
for a, (k, v) in zip(axs, variants.items()): a.imshow(v, cmap="gray", vmin=0, vmax=1); a.set_title(k, fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

### 4.4. Влияние засветки на детектор пожара (ложные «очаги» в RGB)
Наивный RGB-кандидат в очаги: насыщенное пятно площадью 2-400 px, сохраняющееся ≥ 3 кадров подряд. Меры:
- **thermal_gate** — кандидат подтверждается тепловой аномалией ≥ 8 К;
- **prior_mask** — исключаем пятна, совпадающие с *известными источниками света* (эталонный пролёт той же местности с другой реализацией шума; аналог карты ночной засветки);
- **thermal_only** — только тепловизор (верхняя граница в синтетике; реальные ограничения тепловизора не моделируются).

Для каждого уровня засветки `u`: 15 ночных сцен без огня (ложные тревоги) и 15 с огнём (полнота; интенсивность очага 3-100, окно обнаружения 25 кадров от начала). Один и тот же мир используется для обеих групп.

In [ ]:
MODES = ["rgb", "thermal_gate", "prior_mask", "thermal_only"]
def fire_eval(u, n=15, seed0=0):
    rng = np.random.default_rng(seed0 + int(u * 100)); rows = []
    for i in range(n):
        seed = seed0 + int(u * 100) * 100 + i
        ref = make_sequence(SeqConfig(urban=u, seed=seed, noise_seed=seed + 999))
        known = np.array([binary_dilation(f >= 0.98, iterations=2) for f in ref["y"]])
        I = 10 ** rng.uniform(0.5, 2)
        for has in (False, True):
            s = make_sequence(SeqConfig(urban=u, seed=seed, fire=has, fire_I=I, fire_start=30))
            for m in MODES:
                cnt = [rgb_fire_candidates(s["y"][t], s["T"][t], known[t], mode=m) for t in range(len(s["gain"]))]
                a = persistent(cnt, 3); a[:WARM] = False
                rows.append(dict(u=u, mode=m, fire=has, flag=bool(a[30:55].any()) if has else bool(a.any())))
    return pd.DataFrame(rows)
fe = pd.concat([fire_eval(u, seed0=20000) for u in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)])
FA = fe[~fe.fire].groupby(["u", "mode"]).flag.mean().unstack()[MODES]; REC = fe[fe.fire].groupby(["u", "mode"]).flag.mean().unstack()[MODES]
display(pd.concat({"ложные тревоги": FA, "полнота по очагам": REC}, axis=1).round(2))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
FA.plot(ax=axs[0], marker="o", ylim=(-.02, 1.02)); axs[0].set_title("Ложные тревоги детектора очагов"); axs[0].set_xlabel("уровень засветки u")
REC.plot(ax=axs[1], marker="o", ylim=(-.02, 1.02)); axs[1].set_title("Полнота по реальным очагам"); axs[1].set_xlabel("уровень засветки u")
for a in axs: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
def light_alert(t, lpi_value, uav_id="UAV-001"):
    level = "HIGH" if lpi_value > 0.6 else "MEDIUM" if lpi_value > 0.3 else "LOW"
    return {"alert_id": f"lp-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": {"LOW": 1, "MEDIUM": 2, "HIGH": 3}[level],
            "description": f"LIGHT_POLLUTION_{level}", "sensor_source": "rgb+isp_meta+solar_elev", "confidence": round(0.5 + 0.5 * abs(lpi_value - 0.5) * 2, 2),
            "timestamp": int(t * 1000), "recommended_action": "require_thermal_confirmation_for_rgb_fire_candidates; apply_known_lights_mask"}
light_alert(3.0, 0.7)

## 5. Выводы и ограничения
**Что показал эксперимент** (синтетика; зёрна: обучение 1000+, подбор порога 4000+, финальная оценка 2000+/3000+; 150 безобидных и 150 лазерных последовательностей):

**Угроза №10**
- Правило «скачок насыщения + исключение зон с тепловой аномалией» (R1) даёт полноту ≈ 0,93 при нулевой доле ложных тревог на чистых ночных, городских и дневных сценах; медианная задержка 0 кадров (то есть первый затронутый кадр). Без теплового исключения (R0) очаги огня дают ложные тревоги: 0,17 ночью и 0,92 днём.
- **Солнечный блик - трудный негатив.** R1 ошибается на 0,96 бликовых сцен. Требование мерцания (R2) снижает это до 0,54, но полнота по *непрерывным* лазерам падает с 0,93 до 0,60. Дневной контекст (R3) выигрыша почти не даёт (0,50). Результат определяется допущением генератора (блик не мерцает); без реальных данных о бликах вывод нельзя переносить.
- Одноклассовые модели (Isolation Forest, PCA-автоэнкодер) не превзошли правила: полнота 0,70 и 0,81, AUROC по кадрам 0,89-0,92 против 0,92-0,94 у правил.
- По мощности: полнота 0,57-0,70 при I = 10-30, ≈ 0,90 при I = 100 и 1,0 при I ≥ 1000. Слабые лазеры над засвеченным городом обнаруживаются хуже всего.
- **Предотвратить повреждение матрицы ПО не может:** минимальная реакция ≥ одного кадра (200 мс при 5 Гц) плюс время срабатывания механизма защиты. Результат - обнаружение и ограничение последствий.
- Смягчение: маска насыщенных пикселей почти не помогает (усиление во время лазера 0,77 от исходного против 0,73 при штатной AE), так как автоэкспозицию «сжимает» светлый ореол; маска с расширением на 12 px даёт 0,85, **блокировка усиления после тревоги - 0,95**.

**Угроза №16**
- Индекс засветки по экспозиции (LPI_g, без обучения) сохраняет ранжирование уровней засветки при смене оптики (Spearman 0,986 в обоих доменах). GBM при смене домена деградирует: R² 0,98 → 0,36, MAE 0,03 → 0,21. Абсолютная шкала LPI смещается (при u = 1: 0,64 → 0,97), поэтому его следует использовать как порядковый уровень с калибровкой `g_dark` на конкретном аппарате.
- Улучшение ночного кадра платит шумом: CLAHE увеличивает σ шума примерно в 3,1 раза (контраст/шум 2,2 → 1,4), SSR - в 2,1 раза (1,7). Усиление перед детектором без шумоподавления не рекомендуется.
- Наивный RGB-детектор очагов даёт ложные тревоги в **100 %** ночных сцен при любом ненулевом уровне засветки. Тепловое подтверждение и карта известных источников света устраняют их (0 ложных) при полноте 0,93-1,00 (по 15 сцен на уровень: один пропуск = 0,07).

**Ограничения**
- Синтетический одноканальный кадр: цвет, дифракционные лучи, реальный ореол оптики и зеркальные блики не моделируются; автоэкспозиция и PSF заданы нами.
- Идеальная регистрация RGB и тепловизора, идеальная эталонная карта огней: реальные ошибки привязки снизят эффект thermal_gate/prior_mask.
- Тепловизор в синтетике без собственных ложных тревог (нагретые солнцем объекты) - оценка «thermal_only» является верхней границей.
- Метаданные усиления/экспозиции ISP считаются доступными (уточнить у заказчика).
- Малые выборки в экспериментах с очагами (15 на уровень) и по мощности (30 на уровень): доверительные интервалы широкие.

**Дальше:** проверить на реальных ночных парах RGB/ИК (например, LLVIP) и на реальных кадрах с бликами; ввести реальную регистрацию и шум; перейти к ноутбуку 4 (аэрозольная деградация видимости: дым, пыль, мгла).